# Satellite-Derived Chlorophyll Time Series for Lakes: MODIS 250m 8-Day Composite

## Overview

This notebook extracts chlorophyll-a concentration estimates using a NIR/Red ratio algorithm from MODIS Aqua 8-day composite data at 250m resolution for specified lake locations.

The MYD09Q1 product provides Aqua MODIS surface reflectance for bands 1 and 2 at 250m resolution, composited over 8-day periods and corrected for atmospheric conditions such as gases, aerosols, and Rayleigh scattering.

- **Purpose**: Generate time series of chlorophyll indices from high-resolution satellite imagery
- **Study Areas**: Detroit Lake and Upper Klamath Lake
- **Satellite Sensor**: MODIS-Aqua only (no Terra equivalent at 250m)
- **Algorithm**: Two-band NIR/Red ratio with multiple indices
- **Resolution**: 250m (4x higher resolution than 500m products)
- **Temporal**: 8-day composites (reduced temporal but increased spatial resolution)
- **Output**: CSV files with date-stamped chlorophyll index values

## Algorithm Background

The NIR/Red algorithm is specifically designed for turbid, productive waters (Case 2) where traditional blue-green algorithms fail due to interference from suspended sediments and colored dissolved organic matter (CDOM). The algorithm exploits:

- **Band 1 (Red, 620-670nm)**: Chlorophyll absorption maximum
- **Band 2 (NIR, 841-876nm)**: Baseline correction, minimal CDOM/sediment effects

## Advantages of 250m Resolution

- **Higher spatial detail**: 4x better resolution than 500m products
- **Reduced mixed pixels**: Better separation at shorelines
- **Spatial heterogeneity**: Ability to detect within-lake variations
- **Cloud-free composites**: 8-day compositing reduces cloud contamination

## Trade-offs

- **Temporal resolution**: 8-day composites vs daily observations
- **Single satellite**: Aqua only (no Terra comparison)
- **Limited spectral**: Only 2 bands (Red + NIR)

**Key References:**
- Gitelson et al. (2008): Simple semi-analytical model for remote estimation of chlorophyll-a in turbid waters
- Moses et al. (2009): NIR/Red algorithms for turbid waters
- Binding et al. (2012): MODIS-derived algal turbidity in lakes

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

In [2]:
# -----------------------------
# User parameters
# -----------------------------

lakes = [
    dict(
        name='Detroit',
        lon=-122.184, lat=44.711,
        aqua_export='Detroit_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m',
        # Set to None to skip Chl computation (export indices only)
        a=None, b=None
    ),
    dict(
        name='UpperKlamath',
        lon=-121.900, lat=42.400,
        aqua_export='UKL_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m',
        a=None, b=None
    ),
]

start_date = '2011-01-01'  # Shortened date range for testing
end_date   = '2024-12-31'

# If a lake has a/b=None, Chl-a won't be computed (only indices exported)
DEFAULT_A = None
DEFAULT_B = None

# ROI & masking (adjusted for higher resolution)
ROI_RADIUS_M         = 750     # Increased to 750m for more pixels
SHORELINE_BUFFER_M   = 125     # Reduced buffer for testing
WATER_OCC_THRESHOLD  = 50      # Lowered threshold for testing
GSW_DATASET_ID       = 'JRC/GSW1_4/GlobalSurfaceWater'

# MODIS collection and bands (250m 8-day composite)
AQUA_COL_ID  = 'MODIS/061/MYD09Q1'  # Aqua only - no Terra equivalent
B_RED        = 'sur_refl_b01'        # Band 1: ~645 nm (Red, 250 m)
B_NIR        = 'sur_refl_b02'        # Band 2: ~859 nm (NIR, 250 m)
SR_SCALE     = 1e-4                  # scale factor

In [3]:
# First, let's test data availability without any masking
print("Testing MYD09Q1 data availability...")

# Test collection
test_collection = ee.ImageCollection(AQUA_COL_ID)
print(f"Total images in collection: {test_collection.size().getInfo()}")

# Test with date filter
test_filtered = test_collection.filterDate(start_date, end_date)
print(f"Images in date range {start_date} to {end_date}: {test_filtered.size().getInfo()}")

# Test with spatial filter for Detroit Lake
detroit_point = ee.Geometry.Point([-122.184, 44.711])
detroit_roi = detroit_point.buffer(1000)
test_spatial = test_filtered.filterBounds(detroit_roi)
print(f"Images covering Detroit Lake: {test_spatial.size().getInfo()}")

# Get first image info
if test_spatial.size().getInfo() > 0:
    first_img = test_spatial.first()
    print(f"First image date: {ee.Date(first_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()}")
    print(f"Band names: {first_img.bandNames().getInfo()}")
else:
    print("No images found for Detroit Lake!")

Testing MYD09Q1 data availability...
Total images in collection: 1062
Images in date range 2011-01-01 to 2024-12-31: 643
Images covering Detroit Lake: 643
First image date: 2011-01-01
Band names: ['sur_refl_b01', 'sur_refl_b02', 'State', 'QA']


In [4]:
# -----------------------------
# Simplified processing functions
# -----------------------------

def build_water_mask():
    """
    Persistent open-water mask from JRC Global Surface Water 'occurrence'.
    Simplified for testing.
    """
    gsw = ee.Image(GSW_DATASET_ID).select('occurrence')
    water = gsw.gte(WATER_OCC_THRESHOLD)
    # Minimal erosion for testing
    water_eroded = water.focal_min(radius=SHORELINE_BUFFER_M, units='meters')
    return water_eroded

WATER_MASK = build_water_mask()

def per_image_stats(img, roi_geom, sensor_tag, a_coeff, b_coeff):
    """
    Simplified processing with minimal masking for testing.
    """
    # Apply only water mask for now (no QA masking)
    img_masked = img.updateMask(WATER_MASK)

    red = img_masked.select(B_RED).multiply(SR_SCALE)
    nir = img_masked.select(B_NIR).multiply(SR_SCALE)

    # Valid pixel mask - check for reasonable values
    valid = red.gt(0).And(red.lt(1)).And(nir.gt(0)).And(nir.lt(1))
    
    # NIR/Red ratio (primary algorithm for turbid waters)
    nir_red_ratio = nir.divide(red).updateMask(valid).rename('nir_red_ratio')
    
    # NDCI (Normalized Difference Chlorophyll Index)
    ndci = nir.subtract(red).divide(nir.add(red)).updateMask(valid).rename('ndci')
    
    # Log-transformed NIR/Red ratio for potential calibration
    log_nir_red = nir_red_ratio.log10().rename('log_nir_red')

    # Reduce indices over ROI at 250m resolution
    nir_red_stats = nir_red_ratio.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=250,
        maxPixels=1e9,
        bestEffort=True
    )
    ndci_stats = ndci.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=250,
        maxPixels=1e9,
        bestEffort=True
    )
    log_nir_red_stats = log_nir_red.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=250,
        maxPixels=1e9,
        bestEffort=True
    )

    props = ee.Dictionary({
        'datetime': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss'),
        'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
        'time': ee.Date(img.get('system:time_start')).format('HH:mm:ss'),
        'sensor': sensor_tag,
        'composite_day': ee.Date(img.get('system:time_start')).getRelative('day', 'year'),
        'nir_red_ratio': nir_red_stats.get('nir_red_ratio'),
        'ndci': ndci_stats.get('ndci'),
        'log_nir_red': log_nir_red_stats.get('log_nir_red')
    })

    # Optionally compute Chl-a from calibrated relationship
    def add_chl_props(p):
        log10_chl = log_nir_red.multiply(a_coeff).add(b_coeff)
        chl_img = ee.Image(10).pow(log10_chl).rename('chlor_a')
        chl_stats = chl_img.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=roi_geom,
            scale=250,
            maxPixels=1e9,
            bestEffort=True
        )
        return ee.Dictionary(p).set('chlor_a', chl_stats.get('chlor_a'))

    if (a_coeff is not None) and (b_coeff is not None):
        props = add_chl_props(props)

    return ee.Feature(None, props)

def imagecollection_to_features(col_id, roi_geom, sensor_tag, a_coeff, b_coeff):
    ic = (ee.ImageCollection(col_id)
          .filterDate(start_date, end_date)
          .filterBounds(roi_geom))

    fc = ic.map(lambda img: per_image_stats(img, roi_geom, sensor_tag, a_coeff, b_coeff))
    # Don't filter for null values initially to see what we get
    return fc

In [5]:
# --------------------------------------
# Main loop with client-side CSV export
# --------------------------------------

for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(ROI_RADIUS_M)

    a = lake.get('a', DEFAULT_A)
    b = lake.get('b', DEFAULT_B)

    # Only Aqua data available for MYD09Q1
    aqua_fc = imagecollection_to_features(AQUA_COL_ID, roi, 'Aqua_250m_8day', a, b)

    # Availability
    total_features = aqua_fc.size().getInfo()
    print(f"{lake['name']} Aqua 250m 8-day features = {total_features}")
    
    if total_features > 0:
        # Filter for non-null nir_red_ratio
        valid_fc = aqua_fc.filter(ee.Filter.notNull(['nir_red_ratio']))
        valid_count = valid_fc.size().getInfo()
        print(f"{lake['name']} valid features (non-null) = {valid_count}")

        # ---------- Client-side export ----------
        aqua_rows = valid_fc.getInfo()['features']
        aqua_records = [f['properties'] for f in aqua_rows]
        aqua_df = pd.DataFrame.from_records(aqua_records)
        
        if not aqua_df.empty:
            aqua_df = aqua_df.sort_values('datetime') if 'datetime' in aqua_df.columns else aqua_df
            aqua_df.to_csv(lake['aqua_export'] + '.csv', index=False)
            print(f"Exported: {lake['aqua_export']}.csv with {len(aqua_df)} records")
            
            # Show sample of data
            print(f"Sample data for {lake['name']}:")
            print(aqua_df[['date', 'nir_red_ratio', 'ndci']].head())
        else:
            print(f"No valid data for {lake['name']} after filtering")
    else:
        print(f"No images found for {lake['name']}")
    
    print(f"Completed processing for {lake['name']} Lake\n")

print("Processing complete!")

Detroit Aqua 250m 8-day features = 643
Detroit valid features (non-null) = 642
Exported: Detroit_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m.csv with 642 records
Sample data for Detroit:
         date  nir_red_ratio      ndci
0  2011-01-01       1.027149  0.013393
1  2011-01-09       1.358787  0.152106
2  2011-01-17       0.503198 -0.330496
3  2011-01-25       0.261236 -0.585746
4  2011-02-02       0.906542 -0.049020
Completed processing for Detroit Lake

UpperKlamath Aqua 250m 8-day features = 643
UpperKlamath valid features (non-null) = 642
Exported: UKL_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_250m.csv with 642 records
Sample data for UpperKlamath:
         date  nir_red_ratio      ndci
0  2011-01-01       1.001800  0.000899
1  2011-01-09       0.969511 -0.015481
2  2011-01-17       0.140426 -0.753731
3  2011-01-25       0.226721 -0.630363
4  2011-02-02       0.474667 -0.356239
Completed processing for UpperKlamath Lake

Processing complete!
